In [ ]:
import torch
from colpali_engine.models import ColIdefics3, ColIdefics3Processor
from transformers import AutoModel, AutoProcessor, AutoTokenizer
from datasets import load_dataset
from tqdm import tqdm

from transformers.utils.import_utils import is_flash_attn_2_available
from colpali_engine.models import ColQwen2, ColQwen2Processor

from torch.utils.data import DataLoader

import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from umap import UMAP
from matplotlib.patches import Patch
from scipy.spatial.distance import pdist, squareform
from sklearn.neighbors import NearestNeighbors
import copy
import seaborn as sns
from PIL import Image

from typing import ClassVar, List, Optional, Union




## Jina UMAP visualization

In [ ]:
# Load the model
device = 'mps'
model = AutoModel.from_pretrained('jinaai/jina-clip-v2', trust_remote_code=True, device_map=device)


# Load and process the dataset
ds = load_dataset("vidore/syntheticDocQA_artificial_intelligence_test", split="test[:100]")

ds = ds.map(lambda example, idx: {'id': idx}, with_indices=True)

ds_passages = ds.remove_columns(
    [col for col in ds.column_names if col not in ['image', 'image_hash', 'id']]
)
ds_queries = ds.remove_columns(
    [col for col in ds.column_names if col not in ['query', 'id']]
)

queries = list(ds_queries['query'])
passages = list(ds_passages['image'])

#####

dataloader = DataLoader(
    dataset=queries,
    batch_size=1,
    shuffle=False,
)

query_embeddings = []

with torch.no_grad():
    for batch_query in tqdm(dataloader, desc="Forward pass queries...", leave=False):
        embeddings_query = model.encode_text(batch_query, truncate_dim=128)
        query_embeddings.append(np.array(embeddings_query))

query_embeddings = np.array(query_embeddings)
query_embeddings = query_embeddings[:,0,:]


dataloader = DataLoader(
    dataset=passages,
    batch_size=2,
    shuffle=False,
)

image_embeddings = []

with torch.no_grad():
    for batch_doc in tqdm(passages, desc="Forward pass images...", leave=False):
        embeddings_doc = model.encode_image(batch_doc.resize((512,512)), truncate_dim=128)
        image_embeddings.append(np.array(embeddings_doc))

image_embeddings = np.array(image_embeddings)

#Permuting embeddings:
perm = np.random.permutation(len(query_embeddings))
query_embeddings = [query_embeddings[i] for i in perm]
image_embeddings = [image_embeddings[i] for i in perm]

# Sample data: group1 and group2 are lists of [1, X, 128] tensors
group1 = image_embeddings
group2 = query_embeddings

X_jina = np.concatenate([group1, group2], axis=0)

# Fit UMAP model
umap_model = UMAP(
    n_components=2,
    n_neighbors=5,        
    min_dist=0.01,         
    metric='cosine',  
    random_state=42
)
X_embedded_jina = umap_model.fit_transform(X_jina)


# Plotting

import matplotlib as mpl
mpl.use('pgf')

mpl.rcParams.update({
    "pgf.texsystem": "pdflatex",  # or "xelatex"/"lualatex" if you prefer
    "font.family": "serif",
    "text.usetex": True,
    "pgf.rcfonts": False,
})
labels = np.array([0]*len(group1) + [1]*len(group2))

colors = np.array(['red', 'green'])

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    X_embedded_jina[:, 0], 
    X_embedded_jina[:, 1], 
    c=colors[labels], 
    alpha=1,
    edgecolors='w',
    linewidth=0.5,
    s=100
)

plt.tick_params(axis='both', which='major', labelsize=20)  # Set large font size for tick labels



legend_elements = [
    Patch(facecolor='red', edgecolor='w', label='Image Embeddings'),
    Patch(facecolor='green', edgecolor='w', label='Query Embeddings')
]
plt.legend(handles=legend_elements, loc="best", fontsize=15)




plt.title(f'UMAP Visualization of Jina-Clip Retrieval Scores, n_neighbors:{umap_model.n_neighbors}, min_dist:{umap_model.min_dist}')

plt.grid(alpha=0.2)

metric='cosine'
original_distances = squareform(pdist(X_jina, metric=metric))

# Set number of neighbors to show per point
n_neighbors_to_draw = 1

# Find k nearest neighbors in original space
nn = NearestNeighbors(n_neighbors=n_neighbors_to_draw + 1, metric=metric)  # +1 because point is neighbor to itself
nn.fit(X_jina)
distances, indices = nn.kneighbors(X_jina)

nn_not_image_counter = 0
# Draw lines and annotate distances
for i in range(int(len(X_jina)/2),len(X_jina)):
    my_list = list(indices[i][1:])
    #my_list.append(np.random.randint(0, len(X_jina)))
    for j in my_list:  # skip self (first neighbor)
        #j = np.random.randint(0, len(cos_data))
        if i == j or np.random.randint(0,10) < 0:
            continue
        x1, y1 = X_embedded_jina[i]
        x2, y2 = X_embedded_jina[j]
        dist = original_distances[i, j]
        
        # Draw line between points
        #plt.plot([x1, x2], [y1, y2], 'green', linestyle='--', linewidth=0.2)
        
        # Annotate distance near the midpoint
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        #plt.text(mx, my, f"{dist:.2f}", fontsize=8, color='black', ha='center')

    if not i-100 in my_list:  # i-100 is the corresponding image for query i
        nn_not_image_counter += 1
        x1, y1 = X_embedded_jina[i]
        x2, y2 = X_embedded_jina[i-100]
        dist = original_distances[i, i-100]
        
        # Draw line between points
        #plt.plot([x1, x2], [y1, y2], 'red', linestyle='--', linewidth=0.2)
        
        # Annotate distance near the midpoint
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        #plt.text(mx, my, f"{dist:.2f}", fontsize=8, color='blue', ha='center')

print(f"Number of lines drawn for non-image neighbors: {nn_not_image_counter}")

rand_ind = np.random.randint(0,10_000)
#plt.show()
plt.savefig(f'jina_umap_noQuery_{rand_ind}.pgf')
plt.savefig(f'jina_umap_noQuery_{rand_ind}.pdf')

## CLIP UMAP visualization

In [ ]:
# Load the model
device = 'mps'
model_name = "openai/clip-vit-large-patch14"
model = AutoModel.from_pretrained(model_name).to(device)
processor = AutoProcessor.from_pretrained(model_name, use_fast=True)
tokenizer = AutoTokenizer.from_pretrained(model_name)


# Load and process the dataset
ds = load_dataset("vidore/syntheticDocQA_artificial_intelligence_test", split="test[:100]")

ds = ds.map(lambda example, idx: {'id': idx}, with_indices=True)

ds_passages = ds.remove_columns(
    [col for col in ds.column_names if col not in ['image', 'image_hash', 'id']]
)
ds_queries = ds.remove_columns(
    [col for col in ds.column_names if col not in ['query', 'id']]
)

queries = list(ds_queries['query'])

# Load attack JPEG images
clip_qwen_attack_img = Image.open('../data/attacks/paper/adv_img_paper_non_targeted_a73e554c47b471748d8cbe2dd61ea742.JPEG')  
clip_smolvlm_attack_img = Image.open('../data/attacks/paper/adv_img_paper_non_targeted_741d2dc72b1e5a00bf95ff5ecc3cf726.JPEG') 


passages =  list(ds_passages['image'])# + [clip_qwen_attack_img, clip_smolvlm_attack_img]

dataloader = DataLoader(
    dataset=queries,
    batch_size=1,
    shuffle=False,
)

query_embeddings = []

with torch.no_grad():
    for batch_query in tqdm(dataloader, desc="Forward pass queries...", leave=False):
        embeddings_query = model.get_text_features(**tokenizer(batch_query, return_tensors="pt", truncation=True, padding=True).to(device)) 
        query_embeddings.append(embeddings_query.to('cpu').numpy())

query_embeddings = np.array(query_embeddings)
query_embeddings = query_embeddings[:,0,:]


dataloader = DataLoader(
    dataset=passages,
    batch_size=2,
    shuffle=False,
)

image_embeddings = []

with torch.no_grad():
    for batch_doc in tqdm(passages, desc="Forward pass images...", leave=False):
        image_input_emb = processor(images=batch_doc, return_tensors='pt').to(device)

        embeddings_doc = model.get_image_features(**image_input_emb)
        image_embeddings.append(embeddings_doc.to('cpu').numpy())

image_embeddings = np.array(image_embeddings)
image_embeddings = image_embeddings[:,0,:]


#Permuting embeddings:
perm = np.random.permutation(len(query_embeddings))
query_embeddings = [query_embeddings[i] for i in perm]
image_embeddings[:100] = [image_embeddings[i] for i in perm]

# Sample data: group1 and group2 are lists of [1, X, 128] tensors
group1 = image_embeddings
group2 = query_embeddings

X_clip = np.concatenate([group1, group2], axis=0)


# Fit UMAP model
umap_model = UMAP(
    n_components=2,
    n_neighbors=5,        # Controls local vs global structure balance
    min_dist=0.01,          # Minimum distance between embedded points
    metric='cosine',    # Distance metric (adjust based on data)
    random_state=42
)
X_embedded_clip = umap_model.fit_transform(X_clip)


# Plotting
# Visualization with distinct colors

import matplotlib as mpl
mpl.use('pgf')

mpl.rcParams.update({
    "pgf.texsystem": "pdflatex",  # or "xelatex"/"lualatex" if you prefer
    "font.family": "serif",
    "text.usetex": True,
    "pgf.rcfonts": False,
})
labels = np.array([0]*100 + [1]*100) # + [2, 2])

colors = np.array(['red', 'green', 'purple'])

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    X_embedded_clip[:, 0], 
    X_embedded_clip[:, 1], 
    c=colors[labels], 
    alpha=1,
    edgecolors='w',
    linewidth=0.5,
    s=100
)

plt.tick_params(axis='both', which='major', labelsize=20)  # Set large font size for tick labels



legend_elements = [
    Patch(facecolor='red', edgecolor='w', label='Image Embeddings'),
    Patch(facecolor='green', edgecolor='w', label='Query Embeddings')
]
plt.legend(handles=legend_elements, loc="best", fontsize=15)


plt.title(f'UMAP Visualization of Clip Retrieval Scores, n_neighbors:{umap_model.n_neighbors}, min_dist:{umap_model.min_dist}')

plt.grid(alpha=0.2)


metric='cosine'
original_distances = squareform(pdist(X_clip, metric=metric))

# Set number of neighbors to show per point
n_neighbors_to_draw = 1

# Find k nearest neighbors in original space
nn = NearestNeighbors(n_neighbors=n_neighbors_to_draw + 1, metric=metric)  # +1 because point is neighbor to itself
nn.fit(X_clip)
distances, indices = nn.kneighbors(X_clip)

nn_not_image_counter = 0
# Draw lines and annotate distances
for i in range(int(len(X_clip)/2),len(X_clip)):
    my_list = list(indices[i][1:])
    #my_list.append(np.random.randint(0, len(X_clip)))
    for j in my_list:  # skip self (first neighbor)
        #j = np.random.randint(0, len(cos_data))
        if i == j or np.random.randint(0,10) < 0:
            continue
        x1, y1 = X_embedded_clip[i]
        x2, y2 = X_embedded_clip[j]
        dist = original_distances[i, j]
        
        # Draw line between points
        #plt.plot([x1, x2], [y1, y2], 'green', linestyle='--', linewidth=0.2)
        
        # Annotate distance near the midpoint
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        #plt.text(mx, my, f"{dist:.2f}", fontsize=8, color='black', ha='center')

    if not i-100 in my_list:  # i+100 is the corresponding image for query i
        nn_not_image_counter += 1
        x1, y1 = X_embedded_clip[i]
        x2, y2 = X_embedded_clip[i-100]
        dist = original_distances[i, i-100]
        
        # Draw line between points
        #plt.plot([x1, x2], [y1, y2], 'red', linestyle='--', linewidth=0.2)
        
        # Annotate distance near the midpoint
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        #plt.text(mx, my, f"{dist:.2f}", fontsize=8, color='blue', ha='center')

print(f"Number of lines drawn for non-image neighbors: {nn_not_image_counter}")

rand_ind = np.random.randint(0,10_000)
#plt.show()
plt.savefig(f'clip_umap_noQuery_{rand_ind}.pgf')
plt.savefig(f'clip_umap_noQuery_{rand_ind}.pdf')

## ColPali UMAP Visualization

In [ ]:
# Load the model
from transformers import ColPaliForRetrieval, ColPaliProcessor

model_name = "vidore/colpali-v1.3-hf"
model = ColPaliForRetrieval.from_pretrained(
    model_name,
    torch_dtype=torch.float32,
    device_map="mps",  # or "mps" if on Apple Silicon
).eval()

processor = ColPaliProcessor.from_pretrained(model_name)

In [ ]:

#Load and process the dataset
ds = load_dataset("vidore/syntheticDocQA_artificial_intelligence_test", split="test[:100]")

ds = ds.map(lambda example, idx: {'id': idx}, with_indices=True)

ds_passages = ds.remove_columns(
    [col for col in ds.column_names if col not in ['image', 'image_hash', 'id']]
)
ds_queries = ds.remove_columns(
    [col for col in ds.column_names if col not in ['query', 'id']]
)
#ds_queries = deduplicate_dataset_rows(ds=ds_queries, target_column='query')

queries = list(ds_queries['query'])

dataloader = DataLoader(
    dataset=queries,
    batch_size=1,
    shuffle=False,
    collate_fn=processor.process_queries
)

query_embeddings = []

with torch.no_grad():
    for batch_query in tqdm(dataloader, desc="Forward pass queries...", leave=False):
        batch_query = batch_query.to(model.device)
        embeddings_query = model(**batch_query).embeddings.to("cpu")
        #print(f"input:{batch_query['input_ids'].shape}")
        #print(f"output {embeddings_query.shape}")
        query_embeddings.extend(list(torch.unbind(embeddings_query)))


passages = list(ds_passages['image'])

dataloader = DataLoader(
    dataset=passages,
    batch_size=1,
    shuffle=False,
    collate_fn=processor.process_images
)

image_embeddings = []

with torch.no_grad():
    for batch_doc in tqdm(dataloader, desc="Forward pass images...", leave=False):
        batch_doc = batch_doc.to(torch.float32).to(model.device)
        embeddings_doc = model(**batch_doc).embeddings.to("cpu")
        #print(embeddings_doc.shape)
        image_embeddings.extend(list(torch.unbind(embeddings_doc)))


In [ ]:
# Save the embeddings since with Coli it takes for a while to compute
suffix = 'colpali'
torch.save(query_embeddings, f"query_embeddings_{suffix}.pt")
torch.save(image_embeddings, f"passage_embeddings_{suffix}.pt")

In [ ]:
# Load the saved embeddings
suffix = 'colpali'
query_embeddings = torch.load(f"query_embeddings_{suffix}.pt")
image_embeddings = torch.load(f"passage_embeddings_{suffix}.pt")

In [ ]:

# MaxSim score
def score_retrieval(
        query_embeddings: Union["torch.Tensor", List["torch.Tensor"]],
        passage_embeddings: Union["torch.Tensor", List["torch.Tensor"]],
        batch_size: int = 128,
        output_dtype: Optional["torch.dtype"] = None,
        output_device: Union["torch.device", str] = "cpu",
    ) -> "torch.Tensor":
    
        if len(query_embeddings) == 0:
            raise ValueError("No queries provided")
        if len(passage_embeddings) == 0:
            raise ValueError("No passages provided")

        if query_embeddings[0].device != passage_embeddings[0].device:
            raise ValueError("Queries and passages must be on the same device")

        if query_embeddings[0].dtype != passage_embeddings[0].dtype:
            raise ValueError("Queries and passages must have the same dtype")

        if output_dtype is None:
            output_dtype = query_embeddings[0].dtype    

        scores: List[torch.Tensor] = []

        for i in range(0, len(query_embeddings), batch_size):
            batch_scores: List[torch.Tensor] = []
            batch_queries = torch.nn.utils.rnn.pad_sequence(
                query_embeddings[i : i + batch_size], batch_first=True, padding_value=0
            )
            for j in range(0, len(passage_embeddings), batch_size):
                batch_passages = torch.nn.utils.rnn.pad_sequence(
                    passage_embeddings[j : j + batch_size], batch_first=True, padding_value=0
                )
                unnorm_score = torch.einsum("bnd,csd->bcns", batch_queries, batch_passages).max(dim=3)[0]
                print(f"unnorm_score: {unnorm_score.shape}")
                batch_scores.append(
                    unnorm_score.sum(dim=2)
                )
            scores.append(torch.cat(batch_scores, dim=1).to(output_dtype).to(output_device))

        return torch.cat(scores, dim=0)

In [ ]:
#Permuting embeddings:
perm = np.random.permutation(len(query_embeddings))
query_embeddings = [query_embeddings[i] for i in perm]
image_embeddings = [image_embeddings[i] for i in perm]
all_embeddings_norm = copy.deepcopy(query_embeddings) + copy.deepcopy(image_embeddings)
all_embeddings = copy.deepcopy(query_embeddings) + copy.deepcopy(image_embeddings)


for i in range(len(all_embeddings_norm)):
    all_embeddings_norm[i] = all_embeddings_norm[i]/all_embeddings_norm[i].shape[0]

#Calculate the scores
scores = score_retrieval(all_embeddings_norm, all_embeddings)
scores_sym = 1 - 0.5*(scores + scores.T) + 1e-6

In [ ]:
#Fit UMAP model using precomputed distances
umap_model = UMAP(
    n_components=2,
    n_neighbors=5,        # Controls local vs global structure balance
    min_dist=0.01,          # Minimum distance between embedded points
    metric='precomputed',    # Distance metric (adjust based on data)
    #random_state=42
)
X_embedded_all = umap_model.fit_transform(scores_sym)

In [ ]:
# Plotting
import matplotlib as mpl
mpl.use('pgf')

mpl.rcParams.update({
    "pgf.texsystem": "pdflatex",  # or "xelatex"/"lualatex" if you prefer
    "font.family": "serif",
    "text.usetex": True,
    "pgf.rcfonts": False,
})

labels = np.array([0]*100 + [1]*100)

colors = np.array(['green', 'red'])

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    X_embedded_all[:, 0], 
    X_embedded_all[:, 1], 
    c=colors[labels], 
    alpha=1,
    edgecolors='w',
    linewidth=0.5,
    s=100
)

legend_elements = [
    Patch(facecolor='red', edgecolor='w', label='Image Embeddings'),
    Patch(facecolor='green', edgecolor='w', label='Query Embeddings')
]
plt.legend(handles=legend_elements, loc="best",fontsize=15)

n_neighbors_to_draw = 1

# Find k nearest neighbors in original space
nn = NearestNeighbors(n_neighbors=n_neighbors_to_draw + 1, metric='precomputed')  # +1 because point is neighbor to itself
nn.fit(scores_sym)
distances, indices = nn.kneighbors(scores_sym)

# Draw lines and annotate distances
nn_not_image_counter = 0
for i in range(int(len(all_embeddings)/2)):
    my_list = list(indices[i][1:])
    if len(my_list) == 0:
        print(i)
    if np.random.randint(0,10) < 0:
        my_list.append(np.random.randint(0, len(all_embeddings)))
    for point_ind, j in enumerate(my_list):  # skip self (first neighbor)
        #j = np.random.randint(0, len(cos_data))
        #if i == j or np.random.randint(0,10) < 0:
        #    continue
        x1, y1 = X_embedded_all[i]
        x2, y2 = X_embedded_all[j]
        dist = scores_sym[i, j]
        
        if point_ind > 0:
            line_color = 'red'
            text_color = 'red'
        else:
            line_color = 'gray'
            text_color = 'black'
        # Draw line between points
        #plt.plot([x1, x2], [y1, y2], line_color, linestyle='--', linewidth=0.5)
        
        if np.linalg.norm(np.array([x1, y1]) - np.array([x2, y2]),ord=2) < 0.5:
            step = 0.5
            eps_x = np.random.uniform(-step, step)
            eps_y = np.random.uniform(-step, step)
        else:
            eps_x = 0
            eps_y = 0
        # Annotate distance near the midpoint
        mx, my = (x1 + x2) / 2 + eps_x, (y1 + y2) / 2 + eps_y
        #plt.text(mx, my, f"{dist:.2f}", fontsize=8, color=text_color, ha='center')

    if not i+100 in my_list:  # i+100 is the corresponding image for query i
        nn_not_image_counter += 1
        x1, y1 = X_embedded_all[i]
        x2, y2 = X_embedded_all[i+100]
        dist = scores_sym[i, i+100]
        
        # Draw line between points
        plt.plot([x1, x2], [y1, y2], 'blue', linestyle='--', linewidth=0.5)
        
        # Annotate distance near the midpoint
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        #plt.text(mx, my, f"{dist:.2f}", fontsize=8, color='blue', ha='center')

plt.title(f'UMAP Visualization of ColPali Retrieval Scores, n_neighbors:{umap_model.n_neighbors}, min_dist:{umap_model.min_dist}')
#plt.show()
rand_ind = np.random.randint(0,10_000)


plt.savefig(f'colpali_umap_noQuery_{rand_ind}.pgf')
plt.savefig(f'colpali_umap_noQuery_{rand_ind}.pdf')
print(f'pdf saved with id: {rand_ind}') 
print(f"Number of queries that do not have the corresponding image in the nearest neighbors: {nn_not_image_counter}")

## gme-Qwen2 UMAP Visualization

In [ ]:
#Load the model
from transformers import AutoModelForImageTextToText, AutoProcessor
device = 'mps'
model_name = "Alibaba-NLP/gme-Qwen2-VL-2B-Instruct"

model = AutoModelForImageTextToText.from_pretrained(model_name).to(device).eval()
processor = AutoProcessor.from_pretrained(model_name, use_fast=True)
instruction = "You are a helpful AI model."


In [ ]:
# Load and process the dataset
ds = load_dataset("vidore/syntheticDocQA_artificial_intelligence_test", split="test[:100]")

ds = ds.map(lambda example, idx: {'id': idx}, with_indices=True)

ds_passages = ds.remove_columns(
    [col for col in ds.column_names if col not in ['image', 'image_hash', 'id']]
)
ds_queries = ds.remove_columns(
    [col for col in ds.column_names if col not in ['query', 'id']]
)

queries = list(ds_queries['query'])

dataloader = DataLoader(
    dataset=queries,
    batch_size=1,
    shuffle=False,
)

query_embeddings = []

with torch.no_grad():
    for batch_query in tqdm(dataloader, desc="Forward pass queries...", leave=False):
        assert isinstance(batch_query, list)
        msg = [f'<|im_start|>system\n{instruction}<|im_end|>\n<|im_start|>user\n{q}<|im_end|>\n<|im_start|>assistant\n<|endoftext|>' for q in batch_query]
        inputs = processor(text=msg, return_tensors="pt", padding=True, padding_side="left", truncation=True).to(device)
        outputs = model(**inputs, output_hidden_states=True)
        last_hidden_state = outputs.hidden_states[-1]
        embedding = last_hidden_state[:, -1].contiguous()
        embedding = torch.nn.functional.normalize(embedding, p=2, dim=1)
        query_embeddings.append(embedding)

import torchvision.transforms.v2 as T

passages = list(ds_passages['image'])

dataloader = DataLoader(
    dataset=passages,
    batch_size=1,
    shuffle=False,
)

image_embeddings = []

with torch.no_grad():
    for batch_doc in tqdm(passages, desc="Forward pass images...", leave=False):
        image = [batch_doc]

        if isinstance(image, list):
            image = torch.cat([T.Resize((512,512))(T.PILToTensor()(im)).unsqueeze(0) for im in image], dim=0)
        n_image = 1 if len(image.shape) == 3 else image.shape[0]
        image_text = '<|vision_start|><|image_pad|><|vision_end|>'
        msg = [f'<|im_start|>system\n{instruction}<|im_end|>\n<|im_start|>user\n{image_text}<|im_end|>\n<|im_start|>assistant\n<|endoftext|>' for _ in range(n_image)]
        inputs = processor(text=msg, images=image, return_tensors="pt", padding=True, padding_side="left", truncation=True).to(device)
        outputs = model(**inputs, output_hidden_states=True)
        last_hidden_state = outputs.hidden_states[-1]
        embedding = last_hidden_state[:, -1]
        embedding = torch.nn.functional.normalize(embedding, p=2, dim=1)
        image_embeddings.append(embedding)

In [ ]:
#Save the embeddings
suffix = 'gme-qwen2'
torch.save(query_embeddings, f"query_embeddings_{suffix}.pt")
torch.save(image_embeddings, f"passage_embeddings_{suffix}.pt")

In [ ]:
#Load the embeddings
suffix = 'gme-qwen2'
query_embeddings = torch.load(f"query_embeddings_{suffix}.pt")
image_embeddings = torch.load(f"passage_embeddings_{suffix}.pt")

In [ ]:
# Prepare the data for UMAP
query_embeddings  = [q.cpu() for q in query_embeddings]
image_embeddings = [p.cpu() for p in image_embeddings]

query_embeddings = np.array(query_embeddings)
image_embeddings = np.array(image_embeddings)

X_qwen = np.concatenate([query_embeddings, image_embeddings], axis=0)
X_qwen = X_qwen[:,0,:]

# Fit UMAP model
umap_model = UMAP(
    n_components=2,
    n_neighbors=5,        # Controls local vs global structure balance
    min_dist=0.01,          # Minimum distance between embedded points
    metric='cosine',    # Distance metric (adjust based on data)
    random_state=42
)
X_embedded_qwen = umap_model.fit_transform(X_qwen)



# Plotting
import matplotlib as mpl
mpl.use('pgf')

mpl.rcParams.update({
    "pgf.texsystem": "pdflatex",  # or "xelatex"/"lualatex" if you prefer
    "font.family": "serif",
    "text.usetex": True,
    "pgf.rcfonts": False,
})
labels = np.array([0]*len(image_embeddings) + [1]*len(query_embeddings))

colors = np.array(['green', 'red'])

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    X_embedded_qwen[:, 0], 
    X_embedded_qwen[:, 1], 
    c=colors[labels], 
    alpha=1,
    edgecolors='w',
    linewidth=0.5,
    s=100
)

plt.tick_params(axis='both', which='major', labelsize=20)  # Set large font size for tick labels



legend_elements = [
    Patch(facecolor='red', edgecolor='w', label='Image Embeddings'),
    Patch(facecolor='green', edgecolor='w', label='Query Embeddings')
]
plt.legend(handles=legend_elements, loc="best", fontsize=15)




plt.title(f'UMAP Visualization of Qwen Retrieval Scores, n_neighbors:{umap_model.n_neighbors}, min_dist:{umap_model.min_dist}')

plt.grid(alpha=0.2)


metric='cosine'
original_distances = squareform(pdist(X_qwen, metric=metric))

# Set number of neighbors to show per point
n_neighbors_to_draw = 1

# Find k nearest neighbors in original space
nn = NearestNeighbors(n_neighbors=n_neighbors_to_draw + 1, metric=metric)  # +1 because point is neighbor to itself
nn.fit(X_qwen)
distances, indices = nn.kneighbors(X_qwen)

nn_not_image_counter = 0
# Draw lines and annotate distances
for i in range(int(len(X_qwen)/2)):
    my_list = list(indices[i][1:])
    #my_list.append(np.random.randint(0, len(X_qwen)))
    for j in my_list:  # skip self (first neighbor)
        #j = np.random.randint(0, len(cos_data))
        if i == j or np.random.randint(0,10) < 0:
            continue
        x1, y1 = X_embedded_qwen[i]
        x2, y2 = X_embedded_qwen[j]
        dist = original_distances[i, j]
        
        # Draw line between points
        #plt.plot([x1, x2], [y1, y2], 'green', linestyle='--', linewidth=0.2)
        
        # Annotate distance near the midpoint
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        #plt.text(mx, my, f"{dist:.2f}", fontsize=8, color='black', ha='center')

    if not i+100 in my_list:  # i+100 is the corresponding image for query i
        nn_not_image_counter += 1
        x1, y1 = X_embedded_qwen[i]
        x2, y2 = X_embedded_qwen[i+100]
        dist = original_distances[i, i+100]
        
        # Draw line between points
        plt.plot([x1, x2], [y1, y2], 'blue', linestyle='--', linewidth=0.2)
        
        # Annotate distance near the midpoint
        mx, my = (x1 + x2) / 2, (y1 + y2) / 2
        #plt.text(mx, my, f"{dist:.2f}", fontsize=8, color='blue', ha='center')

print(f"Number of lines drawn for non-image neighbors: {nn_not_image_counter}")

rand_ind = np.random.randint(0,10_000)
#plt.show()
plt.savefig(f'qwen_umap_noQuery_{rand_ind}.pgf')
plt.savefig(f'qwen_umap_noQuery_{rand_ind}.pdf')